# Full-Market LSTM — Does the Result Survive a Seed Change?

One strong backtest can be luck. This notebook repeats the same full-market LSTM strategy with five random initializations. Every run uses the same 34-stock universe, features, dates, architecture, training budget, top-five weighting, weekly rebalance, 0.5% commission, and real EGX30 benchmark. **Only the seed changes.**

In [ ]:
import os, sys, json
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from tradinglab.backtester import run_backtest
from tradinglab.data_feed import DataFeed
from tradinglab.features import build_pooled_dataset, build_pooled_sequences
from tradinglab.ml import predict, train_model
from tradinglab.models import LSTMRegressor
from tradinglab.simulator import PortfolioSimulator
from tradinglab.strategies.predictor import predictions_to_weights

SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 100
HIDDEN = 32
SEQ_LEN = 10
TOP_K = 5
REBALANCE_EVERY = 5
COMMISSION = 0.005
STARTING_CAPITAL = 1_000.0

torch.set_num_threads(1)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Rebuild the identical pooled sequence dataset

The split is by calendar date, never by shuffled pooled rows. Normalization is learned from the training period only.

In [ ]:
feed = DataFeed.from_dir('data/egx')
split_day = int(feed.n_days * 0.70)
X_train, _, _, _ = build_pooled_dataset(feed, split_day)
Xtr_seq, ytr_seq, Xte_seq, yte_seq = build_pooled_sequences(feed, split_day, seq_len=SEQ_LEN)

x_mean, x_std = X_train.mean(0), X_train.std(0)
x_std[x_std < 1e-8] = 1.0
Xtr_seq = ((Xtr_seq - x_mean) / x_std).astype('float32')
Xte_seq = ((Xte_seq - x_mean) / x_std).astype('float32')
simulator = PortfolioSimulator(feed, benchmark='egx30', commission=COMMISSION)

baseline = json.loads(Path('dashboard/data/neural_portfolio.json').read_text(encoding='utf-8'))
mlp_final = baseline['metrics']['mlp']['final_equity']
benchmark_final = baseline['metrics']['benchmark']['final_equity']

print(f'Universe: {feed.n_assets} stocks')
print(f'Training sequences: {len(Xtr_seq):,} | Testing sequences: {len(Xte_seq):,}')
print(f'Fixed comparison levels — MLP: {mlp_final:,.2f} EGP | EGX30: {benchmark_final:,.2f} EGP')

## 2. Train five identical LSTMs—change only the seed

In [ ]:
def periodic_lstm_strategy(model):
    state = {'calls': 0, 'weights': np.zeros(feed.n_assets)}
    def strategy(observation):
        if state['calls'] % REBALANCE_EVERY == 0:
            window = ((observation[:, -SEQ_LEN:, :] - x_mean) / x_std).astype('float32')
            predictions = predict(model, window)
            state['weights'] = predictions_to_weights(predictions, TOP_K)
        state['calls'] += 1
        return state['weights'].copy()
    return strategy

def result_metrics(result):
    returns, equity = result['portfolio_returns'], result['portfolio']
    curve = np.r_[1.0, equity]
    drawdown = curve / np.maximum.accumulate(curve) - 1.0
    volatility = returns.std(ddof=1)
    sharpe = np.sqrt(252) * returns.mean() / volatility if volatility > 0 else np.nan
    weights = result['weights']
    previous = np.vstack([np.zeros(feed.n_assets), weights[:-1]])
    turnover = np.abs(weights - previous).sum(axis=1) / 2.0
    return {
        'Ending EGP': equity[-1] * STARTING_CAPITAL,
        'Total return': equity[-1] - 1.0,
        'Sharpe': sharpe,
        'Max drawdown': drawdown.min(),
        'Rebalances': int(np.count_nonzero(turnover > 1e-12)),
    }

rows, curves = [], {}
for seed in SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    model = LSTMRegressor(Xtr_seq.shape[2], hidden=HIDDEN)
    history = train_model(model, Xtr_seq, ytr_seq, Xte_seq, yte_seq, epochs=EPOCHS, lr=1e-3)
    result = run_backtest(simulator, periodic_lstm_strategy(model), lookback=30, start=split_day)
    stats = result_metrics(result)
    rows.append({'Seed': seed, 'Test MSE': history['test'][-1], **stats})
    curves[seed] = result['portfolio'] * STARTING_CAPITAL
    print(f'Seed {seed}: {stats["Ending EGP"]:,.2f} EGP | Sharpe {stats["Sharpe"]:.3f}')

seed_results = pd.DataFrame(rows).set_index('Seed')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for seed, curve in curves.items():
    axes[0].plot(result['dates'], curve, label=f'Seed {seed}', linewidth=1.3)
axes[0].plot(result['dates'], result['benchmark'] * STARTING_CAPITAL, label='EGX30', color='#d29922', linestyle='--', linewidth=2)
axes[0].set(title='Same LSTM strategy, different seeds', xlabel='Date', ylabel='Portfolio value (EGP)')
axes[0].legend(ncol=2)

axes[1].bar(seed_results.index.astype(str), seed_results['Ending EGP'], color='#bc8cff')
axes[1].axhline(mlp_final, label='MLP', color='#58a6ff', linestyle='--')
axes[1].axhline(benchmark_final, label='EGX30', color='#d29922', linestyle='--')
axes[1].set(title='Ending value by seed', xlabel='Seed', ylabel='Ending EGP')
axes[1].legend()
plt.tight_layout(); plt.show()

display(seed_results.style.format({
    'Test MSE': '{:.8f}', 'Ending EGP': '{:,.2f}', 'Total return': '{:.2%}',
    'Sharpe': '{:.3f}', 'Max drawdown': '{:.2%}', 'Rebalances': '{:.0f}',
}))

positive = int((seed_results['Total return'] > 0).sum())
beat_mlp = int((seed_results['Ending EGP'] > mlp_final).sum())
beat_benchmark = int((seed_results['Ending EGP'] > benchmark_final).sum())
mean_ending = seed_results['Ending EGP'].mean()
std_ending = seed_results['Ending EGP'].std(ddof=1)
print(f'Positive returns: {positive}/{len(SEEDS)} seeds')
print(f'Beat fixed MLP comparison: {beat_mlp}/{len(SEEDS)} seeds')
print(f'Beat EGX30: {beat_benchmark}/{len(SEEDS)} seeds')
print(f'Mean ending value: {mean_ending:,.2f} EGP (standard deviation {std_ending:,.2f} EGP)')
if positive == len(SEEDS):
    print('VERDICT: the sign of the result held up across every tested seed.')
else:
    print('VERDICT: performance is seed-sensitive; one strong run is not reliable evidence by itself.')

robustness_payload = {
    'settings': {
        'seeds': SEEDS, 'epochs': EPOCHS, 'hidden': HIDDEN, 'sequence_length': SEQ_LEN,
        'top_k': TOP_K, 'rebalance_every': REBALANCE_EVERY, 'commission': COMMISSION,
    },
    'runs': [
        {
            'seed': int(seed),
            'test_mse': round(float(row['Test MSE']), 8),
            'final_equity': round(float(row['Ending EGP']), 2),
            'total_return': round(float(row['Total return']), 6),
            'sharpe': round(float(row['Sharpe']), 6),
            'max_drawdown': round(float(row['Max drawdown']), 6),
        }
        for seed, row in seed_results.iterrows()
    ],
    'summary': {
        'positive_runs': positive, 'beat_mlp_runs': beat_mlp,
        'beat_benchmark_runs': beat_benchmark, 'total_runs': len(SEEDS),
        'mean_final_equity': round(float(mean_ending), 2),
        'std_final_equity': round(float(std_ending), 2),
    },
}
robustness_path = Path('dashboard/data/neural_robustness.json')
robustness_path.write_text(json.dumps(robustness_payload, indent=2), encoding='utf-8')
print(f'Saved dashboard seed evidence to {robustness_path}')

## Takeaway

The dashboard can show the best observed model, but this table determines how much confidence that single line deserves. A robust strategy should remain useful across seeds and time windows—not merely produce one attractive equity curve.